In [ ]:
# ============================================================================
# omniASR Igbo Blind Spot Analysis
# ============================================================================
# Author:
# Date:
# Purpose: Systematic evaluation of tonal fidelity in omniASR-CTC-1B for Igbo
#
# This notebook documents blind spots in a state-of-the-art multilingual ASR
# model when processing Igbo, a tonal Niger-Congo language.
#
# Dataset:
# Model:
# ============================================================================

# omniASR-CTC-3B Inference

## Goal: Get the model running and verify Igbo support works

---



---

## Step 1: Choose Your Environment & GPU Availability

**Recommendation:** Start with Google Colab (free T4), fallback to Kaggle (P100) if needed, use Modal only if both fail.

### Why This Order:
- Colab: Fastest setup, good for experimentation
- Kaggle: More stable sessions, better for longer runs
- Modal: Costs money but most reliable

---

### Check GPU

---

In [ ]:
!nvidia-smi

---

## Step 2: Environment Setup (Colab Notebook)

----


---

### Install Dependencies

---

In [ ]:
!pip install omnilingual-asr
!pip install huggingface_hub
!apt-get update && apt-get install -y libsndfile1
!apt-get install -y ffmpeg

In [ ]:
# Uninstall conflicting packages and reinstall with specific versions
!pip uninstall -y torchvision torch
!pip install torch==2.1.0 torchvision==0.16.0
!pip install omnilingual-asr

In [ ]:
import omnilingual_asr
print(f"omnilingual-asr version: {omnilingual_asr.__version__}")

---

### Import Model and Initialize

---

In [ ]:
import os
import shutil
import inspect
import torch
import numpy as np
import pandas as pd
from google.colab import drive
from pydub import AudioSegment
from difflib import SequenceMatcher
from scipy import stats
import matplotlib.pyplot as plt

from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline
from omnilingual_asr.models.wav2vec2_llama.lang_ids import supported_langs

from huggingface_hub import HfApi, create_repo, upload_folder
from google.colab import userdata

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
print("Loading omniASR-CTC-1B... (smaller model, ~2-3 minutes)")
pipeline = ASRInferencePipeline(model_card="omniASR_CTC_1B")
print("Model loaded successfully!")


---

## Step 3: Verify Igbo Support

---

### Check Language Support


In [ ]:
# Check total languages
print(f"Total supported languages: {len(supported_langs)}")

# Check for Igbo
igbo_variants = [lang for lang in supported_langs if 'ibo' in lang.lower()]
print(f"\nIgbo language codes found: {igbo_variants}")

# Common Nigerian language codes to verify
nigerian_langs = ['ibo_Latn', 'eng_Latn', 'yor_Latn', 'hau_Latn', 'fra_Latn']
for lang in nigerian_langs:
    status = "SUPPORTED !!!" if lang in supported_langs else "NOT FOUND !!!"
    print(f"{lang}: {status}")

---

## Step 4: Inference: Test with Sample Audio

---

### Mount drive and list files

---

In [ ]:
drive.mount('/content/drive')
# drive.mount("/content/drive", force_remount=True)

In [ ]:
# List your audio files
print("Looking for audio files in your Drive...")
!find /content/drive/MyDrive -name "*.m4a" -o -name "*.mp3" -o -name "*.wav" | head -20

---

### Transcribe all three test files

---

In [ ]:
# File paths
base_path = " "
files_to_test = [
    ("igbo_clean.m4a", "Aha m bụ [NAME]. Kedu ka ị mere taa? Ọ dị mma."),
    ("igbo_codeswitch.m4a", "I need to go to the market, ka m ga azụ ahịa. Maybe I'll buy some rice and yam."),
    ("igbo_tonal.m4a", "Ọ bụ akụkọ ifo. Agadi nwanyị na-esi nri n'ụlọ.")
]

results = []

for filename, ground_truth in files_to_test:
    m4a_path = os.path.join(base_path, filename)
    wav_filename = filename.replace('.m4a', '.wav')
    wav_path = f"/content/{wav_filename}"

    print(f"\n{'='*60}")
    print(f"FILE: {filename}")
    print(f"{'='*60}")
    print(f"GROUND TRUTH:\n{ground_truth}\n")

    # Convert m4a to wav
    print("Converting to WAV format...")
    audio = AudioSegment.from_file(m4a_path, format="m4a")
    audio = audio.set_frame_rate(16000).set_channels(1)  # 16kHz mono
    audio.export(wav_path, format="wav")
    print(f"Saved as: {wav_path}")

    # Transcribe
    print("Transcribing...")
    transcription = pipeline.transcribe(
        inp=[wav_path],
        lang=["ibo_Latn"]
    )

    model_output = transcription[0]
    print(f"MODEL OUTPUT:\n{model_output}\n")

    results.append({
        'file': filename,
        'ground_truth': ground_truth,
        'model_output': model_output
    })

print(f"\n{'='*60}")
print("SUMMARY - ALL TRANSCRIPTIONS")
print(f"{'='*60}")
for r in results:
    print(f"\n{r['file']}:")
    print(f"  Expected: {r['ground_truth']}")
    print(f"  Got:      {r['model_output']}")

---

### Findings (Quick test for blind spots discovery; Ensure Model works)

---

### 1. **igbo_clean.m4a**
- **Expected:**
- **Got:**
- **Analysis:**

### 2. **igbo_codeswitch.m4a**
- **Expected:**
- **Got:**
- **Blind spots found:**

### 3. **igbo_tonal.m4a**
- **Expected:**
- **Got:**
- **Blind spots found:**

---

## What This Means:

- ...

**Next steps:**
1. Record 7-10 more examples targeting each blind spot category
2. Test if this happens ONLY in Igbo or also in other tonal African languages (Yoruba, Hausa as controls)
3. Create HuggingFace dataset with these examples
4. Write the README explaining why this matters (tonal changes = meaning changes)

---

---

## Step 5: Inference: Deep Dive into each possible blind spot category

---

In [ ]:
# Ground truth data for all 21 recordings
ground_truth_data = [
    ("01_script_names.m4a", "script_hallucination", "personal_names", "ibo_Latn",
     "Aha m bụ Chukwuemeka. Nna m bụ Obiora. Nne m bụ Ngozi."),

    ("02_script_formal.m4a", "script_hallucination", "greetings", "ibo_Latn",
     "Nnọọ. Kedu ka ị mere? Ọ dị mma. Daalụ."),

    ("03_script_numbers.m4a", "script_hallucination", "numeric", "ibo_Latn",
     "Otu, abụọ, atọ, anọ, ise, isii, asaa, asatọ, itoolu, iri."),

    ("04_script_proverb.m4a", "script_hallucination", "idiomatic", "ibo_Latn",
     "Onye aghala nwanne ya."),

    ("05_script_slow.m4a", "script_hallucination", "prosody", "ibo_Latn",
     "Aha m bụ [NAME]. Kedu ka ị mere taa? Ọ dị mma."),

    ("06_tonal_akwa.m4a", "tonal_diacritics", "minimal_pair", "ibo_Latn",
     "Akwa, akwa, akwa. Akwà, akwà, akwà. Àkwà, àkwà, àkwà. Ákwá, ákwá, ákwá."),

    ("07_tonal_oke.m4a", "tonal_diacritics", "minimal_pair", "ibo_Latn",
     "Oke, oke, oke. Òkè, òkè, òkè. Ọkè, ọkè, ọkè."),

    ("08_tonal_dense.m4a", "tonal_diacritics", "high_density", "ibo_Latn",
     "Ọ nà-èrì ọ̀jị̀ n'ụ̀tụ̀tụ̀."),

    ("09_tonal_flat.m4a", "tonal_diacritics", "monotone", "ibo_Latn",
     "O na-eri oji n'ututu."),

    ("10_tonal_yoruba.m4a", "tonal_diacritics", "control_language", "yor_Latn",
     "Kí ló dé, kí ló ṣe lẹ́?"),

    ("11_codeswitch_en2ig.m4a", "code_switching", "en_to_ig", "mixed",
     "I'm going to the ọgbọ today with my ụmụnne."),

    ("12_codeswitch_ig2en.m4a", "code_switching", "ig_to_en", "mixed",
     "M ga-eje shopping taa. M chọrọ credit card m."),

    ("13_codeswitch_alternate.m4a", "code_switching", "sentence_level", "mixed",
     "I need rice. M chọrọ ji. I want yam. M chọrọ akpu."),

    ("14_codeswitch_embedded.m4a", "code_switching", "with_diacritics", "mixed",
     "The ụlọ is beautiful. My ụmụaka are playing."),

    ("15_codeswitch_pidgin.m4a", "code_switching", "control_pidgin", "pcm_Latn",
     "I wan chop rice. Abeg, give me water."),

    ("16_context_places.m4a", "cultural_context", "geographic", "ibo_Latn",
     "M bi na Enugu. Nnà m sị Owerri. M ga Onitsha."),

    ("17_context_food.m4a", "cultural_context", "culinary", "mixed",
     "M na-eri jollof rice na egusi soup. Ọ tọrọ ụtọ."),

    ("18_context_proverb.m4a", "cultural_context", "idiomatic", "ibo_Latn",
     "Ọnwa na-agbanwe, anyanwụ na-agbanwe, ma ala adịghị agbanwe."),

    ("19_context_french.m4a", "cultural_context", "control_language", "fra_Latn",
     "Bonjour, comment ça va? Je m'appelle [NAME]. J'habite à Paris."),

    ("20_context_noise.m4a", "cultural_context", "noise_robustness", "ibo_Latn",
     "Aha m bụ [NAME]. Kedu ka ị mere taa? Ọ dị mma."),

    ("21_tonal_yoruba_formal.m4a", "tonal_diacritics", "control_formal", "yor_Latn",
     "Ẹ káàárọ̀. Báwo ni?")
]

# Process all files
base_path = " "
results = []

for filename, category, subcategory, lang_code, ground_truth in ground_truth_data:
    m4a_path = os.path.join(base_path, filename)
    wav_path = f"/content/{filename.replace('.m4a', '.wav')}"

    print(f"\n{'='*70}")
    print(f"Processing: {filename}")
    print(f"Category: {category} | Subcategory: {subcategory}")
    print(f"{'='*70}")

    # Convert to WAV
    try:
        audio = AudioSegment.from_file(m4a_path, format="m4a")
        audio = audio.set_frame_rate(16000).set_channels(1)
        audio.export(wav_path, format="wav")

        # Transcribe
        lang_param = [lang_code] if lang_code != "mixed" else ["ibo_Latn"]
        transcription = pipeline.transcribe(
            inp=[wav_path],
            lang=lang_param
        )

        model_output = transcription[0]

        print(f"GROUND TRUTH: {ground_truth}")
        print(f"MODEL OUTPUT: {model_output}")

        results.append({
            'filename': filename,
            'category': category,
            'subcategory': subcategory,
            'language': lang_code,
            'ground_truth': ground_truth,
            'model_output': model_output
        })

    except Exception as e:
        print(f"ERROR: {e}")
        results.append({
            'filename': filename,
            'category': category,
            'subcategory': subcategory,
            'language': lang_code,
            'ground_truth': ground_truth,
            'model_output': f"ERROR: {str(e)}"
        })

# Create DataFrame
df = pd.DataFrame(results)

# Save to CSV
csv_path = " "
df.to_csv(csv_path, index=False)
print(f"\n{'='*70}")
print(f"Results saved to: {csv_path}")
print(f"{'='*70}")

# Display summary
print("\n SUMMARY BY CATEGORY:")
print(df.groupby('category').size())

---

### Findings (Systematic Blind Spot Detection)

---
We've systematically documented severe blind spots across all categories. Let's analyse our findings:

---

## **CRITICAL FINDINGS - High-Level Analysis**

### **1. Script Hallucination**


---

### **2. Tonal Diacritics**


---

### **3. Control Languages**


---

### **4. Code-Switching**


---

### **5. Cultural Context**


---

### **6. Noise Robustness (File XX)**



---

**Next steps:**
1. Perform quantitative analysis and calculate error rate

---

## Step 6a: Quantitative Analysis

---

In [ ]:
df = pd.read_csv(csv_path)

def count_diacritics(text):
    """Count diacritic characters in text"""
    diacritics = set('ụọịàèìòùáéíóúẹṣ')
    return sum(1 for c in text.lower() if c in diacritics)

def character_error_rate(ref, hyp):
    """Calculate character-level similarity"""
    return 1 - SequenceMatcher(None, ref.lower(), hyp.lower()).ratio()

# Calculate metrics
df['ground_truth_diacritics'] = df['ground_truth'].apply(count_diacritics)
df['model_output_diacritics'] = df['model_output'].apply(count_diacritics)
df['diacritic_loss'] = df['ground_truth_diacritics'] - df['model_output_diacritics']
df['diacritic_loss_pct'] = (df['diacritic_loss'] / df['ground_truth_diacritics'] * 100).fillna(0)
df['cer'] = df.apply(lambda row: character_error_rate(row['ground_truth'], row['model_output']), axis=1)

# Summary statistics
print("="*70)
print("BLIND SPOT QUANTIFICATION")
print("="*70)
print(f"\n DIACRITIC LOSS:")
print(f"  Total diacritics in ground truth: {df['ground_truth_diacritics'].sum()}")
print(f"  Total diacritics in model output: {df['model_output_diacritics'].sum()}")
print(f"  Net loss: {df['diacritic_loss'].sum()} ({df['diacritic_loss'].sum() / df['ground_truth_diacritics'].sum() * 100:.1f}%)")

print(f"\n BY CATEGORY:")
category_stats = df.groupby('category').agg({
    'diacritic_loss': 'sum',
    'ground_truth_diacritics': 'sum',
    'cer': 'mean'
})
category_stats['diacritic_loss_pct'] = (category_stats['diacritic_loss'] / category_stats['ground_truth_diacritics'] * 100)
print(category_stats)

print(f"\n WORST PERFORMERS (by CER):")
print(df.nlargest(5, 'cer')[['filename', 'category', 'cer', 'diacritic_loss']])

print(f"\n Detailed results saved to CSV")

# Save enhanced CSV
analysis_path = " "
df.to_csv(analysis_path, index=False)

In [ ]:
# Binomial exact CI for overall DRR
successes = df['model_output_diacritics'].sum()  # diacritics preserved
trials = df['ground_truth_diacritics'].sum()     # total diacritics expected

ci_low, ci_high = stats.binom.interval(0.95, trials, successes/trials)
drr_ci_low = ci_low / trials
drr_ci_high = ci_high / trials

print(f"Overall DRR: {successes/trials:.3f}")
print(f"95% CI: [{drr_ci_low:.3f}, {drr_ci_high:.3f}]")

# Category-specific (for tonal)
tonal_preserved = df[df['category'] == 'tonal_diacritics']['model_output_diacritics'].sum()
tonal_total = df[df['category'] == 'tonal_diacritics']['ground_truth_diacritics'].sum()

tonal_ci_low, tonal_ci_high = stats.binom.interval(0.95, tonal_total, tonal_preserved/tonal_total)
tonal_drr_ci_low = tonal_ci_low / tonal_total
tonal_drr_ci_high = tonal_ci_high / tonal_total

print(f"\nTonal DRR: {tonal_preserved/tonal_total:.3f}")
print(f"95% CI: [{tonal_drr_ci_low:.3f}, {tonal_drr_ci_high:.3f}]")

---

### **KEY QUANTITATIVE FINDINGS** (Quantitative Anaysis)

---

### **1. Overall Diacritic Loss: 26.8%**


### **2. Category Breakdown (CRITICAL):**

**Tonal Diacritics: XX.X% LOSS**

**Script Hallucination: XX.X% (ADDED diacritics!)**

**Code-Switching: XX.X% loss**

**Cultural Context: XX.X% loss**

---

### **3. Worst Individual Files:**

**#1: File A - CER XX.X%**

**#2: File B - CER XX.X%**

**#3: File C - CER XX.X%**

---

**DRR Calculations:**

**From this data:**
- Overall DRR (preserved) =
- Tonal DRR (preserved) =
- Code-switching DRR =
- Cultural DRR =



**Diacritic Retention Rate (DRR):**
- Overall: XX.X% (../.. diacritics preserved)
- 95% CI: [XX.X%, XX.X%] (binomial exact)

**Category-Specific DRR:**
| Category | DRR | 95% CI |
|----------|-----|--------|
| Phonemic Tone Sensitivity | XX.X% | [XX.X%, XX.X%] |
| Language Boundary Effects | XX.X% | [XX.X%, XX.X%] |
| Domain-Specific Lexical Coverage | XX.X% | [XX.X%, XX.X%] |
| Cross-lingual Orthographic Interference | N/A | (net addition) |

**Interpretation:**

### Diacritic-Specific Metrics

Standard Character Error Rate (CER) conflates spacing, capitalization, and tonal errors. We define **Diacritic Error Rate (DER)** to isolate tone-related failures:

$$
\text{DER} = \frac{\text{diacritics_lost} + \text{diacritics_hallucinated}}{\text{diacritics_expected}}
$$

**Results:**
- Overall DER: XX.X%  (vs. CER: XX.X% )
- Phonemic Tone Sensitivity DER: XX.X%  (vs. CER: XX.X% )

**Why DER matters:** In tonal languages, diacritic errors change word meaning for same spelled word. DER quantifies semantic preservation failure independent of general transcription accuracy.


---


---

## Step 6b: Statistical Analysis

---

In [ ]:
# ========================================================================
# STATISTICAL RIGOR: Bootstrap Confidence Intervals
# ========================================================================

df = pd.read_csv(analysis_path)

# Bootstrap CI function
def bootstrap_ci(data, stat_fn, n_boot=10000, ci=0.95, seed=42):
    """Resample rows with replacement, compute CI."""
    rng = np.random.default_rng(seed)
    n = len(data)
    if n == 0:
        return (np.nan, np.nan, np.nan)

    point = float(stat_fn(data))
    boots = np.empty(n_boot, dtype=np.float64)

    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boots[i] = float(stat_fn(data.iloc[idx]))

    alpha = (1 - ci) / 2
    lo = float(np.quantile(boots, alpha))
    hi = float(np.quantile(boots, 1 - alpha))
    return point, lo, hi

# Define statistics
def stat_loss_rate(d):
    exp = d['ground_truth_diacritics'].sum()
    drops = d['diacritic_loss'].clip(lower=0).sum()
    return drops / exp if exp > 0 else np.nan

def stat_halluc_rate(d):
    prod = d['model_output_diacritics'].sum()
    halluc = (-d['diacritic_loss']).clip(lower=0).sum()  # negative loss = hallucination
    return halluc / prod if prod > 0 else np.nan

def stat_avg_cer(d):
    return d['cer'].mean()

# Compute CIs
print("="*70)
print("BOOTSTRAP CONFIDENCE INTERVALS (10,000 resamples)")
print("="*70)

# Overall
overall_loss = bootstrap_ci(df, stat_loss_rate)
overall_halluc = bootstrap_ci(df, stat_halluc_rate)
overall_cer = bootstrap_ci(df, stat_avg_cer)

print(f"\nOverall diacritic loss: {overall_loss[0]*100:.1f}% (95% CI: [{overall_loss[1]*100:.1f}%, {overall_loss[2]*100:.1f}%])")
print(f"Overall hallucination rate: {overall_halluc[0]*100:.1f}% (95% CI: [{overall_halluc[1]*100:.1f}%, {overall_halluc[2]*100:.1f}%])")
print(f"Overall CER: {overall_cer[0]:.3f} (95% CI: [{overall_cer[1]:.3f}, {overall_cer[2]:.3f}])")

# Tonal category
tonal = df[df['category'] == 'tonal_diacritics']
tonal_loss = bootstrap_ci(tonal, stat_loss_rate)
tonal_cer = bootstrap_ci(tonal, stat_avg_cer)

print(f"\nTonal category loss: {tonal_loss[0]*100:.1f}% (95% CI: [{tonal_loss[1]*100:.1f}%, {tonal_loss[2]*100:.1f}%])")
print(f"Tonal category CER: {tonal_cer[0]:.3f} (95% CI: [{tonal_cer[1]:.3f}, {tonal_cer[2]:.3f}])")

# Script category
script = df[df['category'] == 'script_hallucination']
script_halluc = bootstrap_ci(script, stat_halluc_rate)

print(f"\nScript hallucination rate: {script_halluc[0]*100:.1f}% (95% CI: [{script_halluc[1]*100:.1f}%, {script_halluc[2]*100:.1f}%])")

print("\n" + "="*70)
print("INTERPRETATION")
print("="*70)
print("Even with small sample size (N=21), confidence intervals show:")

---

## Step 6c: Visualizations

---



---

### Plot 1: Diacritic Loss by Category (Bar Chart with Error Bars)
---
Shows the dramatic difference between tonal vs. other categories


In [ ]:
# Aggregate by category
category_stats = df.groupby('category').agg({
    'diacritic_loss': 'sum',
    'ground_truth_diacritics': 'sum'
}).reset_index()

category_stats['loss_pct'] = (category_stats['diacritic_loss'] / category_stats['ground_truth_diacritics'] * 100).fillna(0)

# Sort by loss percentage
category_stats = category_stats.sort_values('loss_pct', ascending=False)

# Create plot
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#d62728' if x > 50 else '#ff7f0e' if x > 20 else '#2ca02c' for x in category_stats['loss_pct']]
bars = ax.barh(category_stats['category'], category_stats['loss_pct'], color=colors, alpha=0.8, edgecolor='black')

# Add value labels
for i, (cat, val) in enumerate(zip(category_stats['category'], category_stats['loss_pct'])):
    ax.text(val + 2, i, f'{val:.1f}%', va='center', fontsize=11, fontweight='bold')

ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_xlabel('Diacritic Loss Rate (%)', fontsize=12, fontweight='bold')
ax.set_title('Diacritic Loss Rate by Error Category\n(Positive = Loss, Negative = Hallucination)',
             fontsize=14, fontweight='bold', pad=20)
ax.grid(axis='x', alpha=0.3, linestyle='--')
ax.set_xlim(-50, 70)

# Color legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#d62728', alpha=0.8, label='Severe (>50%)'),
    Patch(facecolor='#ff7f0e', alpha=0.8, label='Moderate (20-50%)'),
    Patch(facecolor='#2ca02c', alpha=0.8, label='Low (<20%)')
]
ax.legend(handles=legend_elements, loc='upper right', fontsize=10)

plt.tight_layout()
fig1_path = " "
plt.savefig(fig1_path, dpi=300, bbox_inches='tight')
plt.show()

print("Figure 1 saved: Diacritic Loss by Category")

---

### Plot 2: Per-Sample CER vs. Diacritic Loss (Scatter Plot)
---

Shows relationship between general transcription errors and tonal errors


In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

# Color by category
category_colors = {
    'tonal_diacritics': '#d62728',
    'script_hallucination': '#ff7f0e',
    'code_switching': '#2ca02c',
    'cultural_context': '#1f77b4'
}

for category in df['category'].unique():
    subset = df[df['category'] == category]
    ax.scatter(subset['cer'] * 100, subset['diacritic_loss_pct'],
              label=category.replace('_', ' ').title(),
              s=120, alpha=0.7, edgecolors='black', linewidth=1.5,
              color=category_colors.get(category, '#gray'))

ax.axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.5, label='No diacritic error')
ax.axhline(y=50, color='red', linestyle=':', linewidth=1.5, alpha=0.7, label='50% loss threshold')

ax.set_xlabel('Character Error Rate (%)', fontsize=12, fontweight='bold')
ax.set_ylabel('Diacritic Loss Rate (%)', fontsize=12, fontweight='bold')
ax.set_title('Sample-Level CER vs. Diacritic Loss\n(Each point = one audio file)',
             fontsize=14, fontweight='bold', pad=20)
ax.grid(alpha=0.3, linestyle='--')
ax.legend(loc='upper left', fontsize=9, framealpha=0.9)

plt.tight_layout()
fig2_path = " "
plt.savefig(fig2_path, dpi=300, bbox_inches='tight')
plt.show()

print("Figure 2 saved: CER vs. Diacritic Loss")

---

###  Plot 3: Bootstrap CI Visualization (Forest Plot)
---
Shows statistical rigor - uncertainty quantification


In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

# Compute CIs for each category
categories = df['category'].unique()
results = []

for cat in categories:
    subset = df[df['category'] == cat]
    if len(subset) > 0:
        loss_ci = bootstrap_ci(subset, stat_loss_rate, n_boot=5000)
        results.append({
            'category': cat.replace('_', ' ').title(),
            'point': loss_ci[0] * 100,
            'lower': loss_ci[1] * 100,
            'upper': loss_ci[2] * 100
        })

# Add overall
overall_ci = bootstrap_ci(df, stat_loss_rate, n_boot=5000)
results.append({
    'category': 'OVERALL',
    'point': overall_ci[0] * 100,
    'lower': overall_ci[1] * 100,
    'upper': overall_ci[2] * 100
})

# Plot
y_positions = range(len(results))
for i, r in enumerate(results):
    color = '#d62728' if r['point'] > 50 else '#ff7f0e' if r['point'] > 20 else '#2ca02c'
    if r['category'] == 'OVERALL':
        color = '#8c564b'
        linewidth = 3
        markersize = 12
    else:
        linewidth = 2
        markersize = 10

    # Error bar
    ax.plot([r['lower'], r['upper']], [i, i], color=color, linewidth=linewidth, alpha=0.8)
    # Point estimate
    ax.plot(r['point'], i, 'o', color=color, markersize=markersize, markeredgecolor='black', markeredgewidth=1.5)

    # Label
    ax.text(-5, i, r['category'], ha='right', va='center', fontsize=11, fontweight='bold')
    ax.text(r['upper'] + 2, i, f"{r['point']:.1f}%", va='center', fontsize=9)

ax.axvline(x=0, color='black', linestyle='-', linewidth=1)
ax.axvline(x=50, color='red', linestyle=':', linewidth=1.5, alpha=0.5, label='50% threshold')

ax.set_xlabel('Diacritic Loss Rate (%) with 95% Bootstrap CI', fontsize=12, fontweight='bold')
ax.set_title('Bootstrap Confidence Intervals for Diacritic Loss\n(10,000 resamples per category)',
             fontsize=14, fontweight='bold', pad=20)
ax.set_ylim(-0.5, len(results) - 0.5)
ax.set_xlim(-10, 85)
ax.set_yticks([])
ax.grid(axis='x', alpha=0.3, linestyle='--')
ax.legend(loc='lower right', fontsize=10)

plt.tight_layout()
fig3_path = " "
plt.savefig(fig3_path, dpi=300, bbox_inches='tight')
plt.show()

print("Figure 3 saved: Bootstrap Confidence Intervals")

---

### Plot 4:  Minimal Pairs Confusion Matrix
---

Visually shows the model can't distinguish tones


In [ ]:
# Extract minimal pair data (files 06 and 07)
minimal_pairs = df[df['filename'].str.contains('06_tonal_akwa|07_tonal_oke')]

if len(minimal_pairs) > 0:
    fig, ax = plt.subplots(figsize=(8, 6))

    for _, row in minimal_pairs.iterrows():
        ax.text(0.1, 0.8, f"File: {row['filename']}", fontsize=12, fontweight='bold', transform=ax.transAxes)
        ax.text(0.1, 0.7, f"Expected:\n{row['ground_truth']}", fontsize=10, transform=ax.transAxes,
               bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.5))
        ax.text(0.1, 0.5, f"Got:\n{row['model_output']}", fontsize=10, transform=ax.transAxes,
               bbox=dict(boxstyle='round', facecolor='lightcoral', alpha=0.5))
        ax.text(0.1, 0.3, f"CER: {row['cer']*100:.1f}%", fontsize=11, transform=ax.transAxes, fontweight='bold')
        break  # Just show first example

    ax.axis('off')
    ax.set_title('Minimal Pair Failure Example\n(Tonal distinctions collapsed)', fontsize=14, fontweight='bold')

    plt.tight_layout()
    fig4_path = " "
    plt.savefig(fig4_path, dpi=300, bbox_inches='tight')
    plt.show()

    print("Figure 4 saved: Minimal Pair Example")

---

## Step 7: Create HuggingFace Dataset Files

---

In [ ]:
# Create dataset directory structure
dataset_dir = " "
audio_dir = os.path.join(dataset_dir, "audio")
os.makedirs(audio_dir, exist_ok=True)

# Copy WAV files to dataset
base_path = " "
for i in range(1, 22):
    src_wav = f"/content/{str(i).zfill(2)}_*.wav"
    # Find the actual file
    import glob
    matching = glob.glob(src_wav)
    if matching:
        filename = os.path.basename(matching[0])
        shutil.copy(matching[0], os.path.join(audio_dir, filename))

# Load detailed analysis
analysis_path = " "
# df = pd.read_csv(f"{base_path}/detailed_analysis.csv")
df = pd.read_csv(analysis_path)

# Create metadata for HuggingFace
metadata = []
for _, row in df.iterrows():
    wav_filename = row['filename'].replace('.m4a', '.wav')
    metadata.append({
        'file_name': f"audio/{wav_filename}",
        'ground_truth': row['ground_truth'],
        'model_output': row['model_output'],
        'category': row['category'],
        'subcategory': row['subcategory'],
        'language': row['language'],
        'character_error_rate': round(row['cer'], 3),
        'diacritics_expected': int(row['ground_truth_diacritics']),
        'diacritics_produced': int(row['model_output_diacritics']),
        'diacritic_loss': int(row['diacritic_loss'])
    })

# Save metadata
metadata_df = pd.DataFrame(metadata)
metadata_df.to_csv(os.path.join(dataset_dir, "metadata.csv"), index=False)

print(f"Dataset created at: {dataset_dir}")
print(f"Files:")
print(f"  - audio/ (21 WAV files)")
print(f"  - metadata.csv")
print(f"\nNext: Create README.md")

---

## Step 8: Create `README.md`

---

In [ ]:
readme_content = r"""  """

In [ ]:
# Save README
readme_path = " "
with open(readme_path, 'w', encoding='utf-8') as f:
    f.write(readme_content)

print("README.md created!")
print(f"Location: {readme_path}")
print("\n README Preview (first 50 lines):")
print("="*70)
for i, line in enumerate(readme_content.split('\n')[:50], 1):
    print(line)
print("\n... (see full file for complete README)")
print("\n READY TO UPLOAD TO HUGGINGFACE")

---

## Step 9: Upload HuggingFace Dataset

---

---

### Login to HuggingFace

---

In [ ]:
# Get token from Colab secrets
try:
    hf_token = userdata.get('HF_TOKEN')
    print("Token loaded from Colab secrets")
except:
    print("HF_TOKEN not found in secrets")
    print("Add it: Click [key] icon in left sidebar → Add 'HF_TOKEN'")
    hf_token = None

# Login with token
if hf_token:
    from huggingface_hub import login
    login(token=hf_token, add_to_git_credential=False)
    print("Logged in to HuggingFace")

---

### B. Create the Dataset Repository

---

In [ ]:
repo_id = "[USERNAME]/omniASR-igbo-blindspots"  # Your username/dataset-name

try:
    create_repo(
        repo_id=repo_id,
        repo_type="dataset",
        private=False  # Make it public so reviewers can see it
    )
    print(f"✅ Created dataset: https://huggingface.co/datasets/{repo_id}")
except Exception as e:
    print(f"Repo might already exist: {e}")

---

### C. Upload Everything

---

In [ ]:
dataset_path = " "

upload_folder(
    folder_path=dataset_path,
    repo_id=repo_id,
    repo_type="dataset",
    commit_message="Initial upload: Igbo blind spot evaluation dataset"
)

print(f"Dataset uploaded!")
print(f"View at: https://huggingface.co/datasets/{repo_id}")

In [ ]:
# Copy figures to dataset folder

figures = [
    fig1_path,
    fig2_path,
    fig3_path
]

for fig in figures:
    if os.path.exists(fig):
        shutil.copy(fig, dataset_path)
        print(f"Copied {os.path.basename(fig)}")

# Then re-upload
upload_folder(
    folder_path=dataset_path,
    repo_id=repo_id,
    repo_type="dataset",
    commit_message="Added visualization figures"
)

---

### D. Verify Upload

---


Go to `https://huggingface.co/datasets/[USERNAME]/omniASR-igbo-blindspots` and check:
- ✅ README.md renders correctly
- ✅ Audio files are in `audio/` folder
- ✅ `metadata.csv` is present
- ✅ All 21 WAV files visible

---